# Supply Chain & Logística Inteligente com IA

## Segmentação de Clientes Logísticos e Detecção de Anomalias Operacionais

**Autor:** Fábio Andrade  
**Disciplina:** Supply Chain & Logística Inteligente com IA  
**Tipo:** Projeto final acadêmico  
**Ambiente de execução:** Google Colab  
**Repositório:** [supply-chain-clustering-anomaly-ai](https://github.com/thedrads/supply-chain-clustering-anomaly-ai)

### Linha escolhida

**Identificação de Clusters e Detecção de Anomalia**

### Objetivo do projeto

Aplicar técnicas de análise de dados e machine learning para segmentar clientes logísticos, identificar comportamentos fora do padrão e gerar insights gerenciais sobre custo logístico, nível de serviço, risco operacional e priorização de clientes.

## Preparação do ambiente

**Objetivo:** importar as bibliotecas necessárias para análise dos dados, visualização, clusterização e detecção de anomalias.

Essas ferramentas serão utilizadas durante todo o pipeline analítico do projeto.

In [4]:
# Importa bibliotecas para manipulação de dados.
import numpy as np
import pandas as pd

# Importa bibliotecas para visualização.
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

# Importa recursos para clusterização e detecção de anomalias.
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Configura a apresentação das tabelas e dos gráficos.
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="deep")

# Define a semente para permitir resultados reproduzíveis.
random_state = 42

## Carregamento da base de dados

**Objetivo:** carregar a base sintética de clientes logísticos diretamente do GitHub.

A leitura pela URL pública torna o notebook reproduzível e elimina a necessidade de Google Drive ou upload manual.

In [5]:
# Define a localização pública da base sintética no GitHub.
dataset_url = (
    "https://raw.githubusercontent.com/thedrads/"
    "supply-chain-clustering-anomaly-ai/main/data/synthetic/"
    "clientes_logisticos_sinteticos_supply_chain.csv"
)

# Carrega os dados diretamente para um DataFrame.
clientes_df = pd.read_csv(dataset_url)

# Confirma as dimensões da base carregada.
print(
    f"Base carregada: {clientes_df.shape[0]} linhas e "
    f"{clientes_df.shape[1]} colunas."
)

Base carregada: 6000 linhas e 23 colunas.


## Visão inicial da base

**Objetivo:** visualizar os primeiros registros para conferir a estrutura, as colunas e os valores da base carregada.

In [6]:
# Exibe os cinco primeiros registros da base.
clientes_df.head()

,cliente_id,regiao,segmento_negocio,canal_relacionamento,porte_cliente,distancia_cd_km,volume_medio_entregue_kg,entregas_mes,ticket_medio_rs,receita_mensal_estimada_rs,lead_time_medio_dias,entregas_no_prazo_pct,custo_medio_entrega_rs,custo_logistico_mensal_rs,custo_por_kg_rs,taxa_ocorrencias_pct,variabilidade_demanda_pct,pedidos_urgentes_pct,prioridade_contrato,risco_operacional_score,anomalia_sintetica,tipo_anomalia_sintetica,cluster_teorico_sintetico
0,CLI_00001,Sul,Varejo,Marketplace,Pequeno,105.50,81.60,4,"2,328.61","9,314.42",3.47,99.11,325.95,"1,303.80",3.99,11.89,50.13,38.27,Baixa,27.32,0,normal,Baixo volume instável
1,CLI_00002,Sudeste,Indústria,Distribuidor,Grande,134.60,584.80,10,"4,243.14","42,431.38",2.83,97.21,280.25,"2,802.50",0.48,5.87,25.38,22.12,Média,18.20,0,normal,Recorrentes padrão
2,CLI_00003,Sudeste,Varejo,Venda direta,Grande,130.90,281.80,7,"1,838.57","12,870.01",3.46,87.31,280.49,"1,963.43",1.00,6.14,49.63,58.62,Baixa,32.70,0,normal,Baixo volume instável
3,CLI_00004,Nordeste,E-commerce,Distribuidor,Pequeno,379.30,917.80,7,"5,833.62","40,835.36",4.88,84.35,707.91,"4,955.37",0.77,6.01,22.65,25.58,Média,31.94,0,normal,Longa distância alto custo
4,CLI_00005,Sudeste,Varejo,B2B,Pequeno,61.10,"1,539.20",19,"10,152.08","192,889.60",1.36,95.39,209.20,"3,974.80",0.14,0.69,7.35,10.15,Alta,8.75,0,normal,Estratégicos urbanos


## Estrutura e tipos dos dados

**Objetivo:** examinar a estrutura do DataFrame, os tipos das variáveis e a quantidade de valores preenchidos em cada coluna.

In [7]:
# Apresenta a estrutura, os tipos e o preenchimento das colunas.
clientes_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   cliente_id                  6000 non-null   object 
 1   regiao                      6000 non-null   object 
 2   segmento_negocio            6000 non-null   object 
 3   canal_relacionamento        6000 non-null   object 
 4   porte_cliente               6000 non-null   object 
 5   distancia_cd_km             6000 non-null   float64
 6   volume_medio_entregue_kg    6000 non-null   float64
 7   entregas_mes                6000 non-null   int64  
 8   ticket_medio_rs             6000 non-null   float64
 9   receita_mensal_estimada_rs  6000 non-null   float64
 10  lead_time_medio_dias        6000 non-null   float64
 11  entregas_no_prazo_pct       6000 non-null   float64
 12  custo_medio_entrega_rs      6000 non-null   float64
 13  custo_logistico_mensal_rs   6000 

## Verificação de integridade

**Objetivo:** verificar valores ausentes, linhas duplicadas e identificadores de clientes repetidos.

Essa validação evita que problemas de qualidade comprometam as análises e os modelos posteriores.

In [8]:
# Conta todos os valores ausentes da base.
total_valores_ausentes = clientes_df.isna().sum().sum()

# Verifica linhas e identificadores duplicados.
total_linhas_duplicadas = clientes_df.duplicated().sum()
total_ids_duplicados = clientes_df["cliente_id"].duplicated().sum()

# Apresenta o resumo da verificação.
print(f"Valores ausentes: {total_valores_ausentes}")
print(f"Linhas duplicadas: {total_linhas_duplicadas}")
print(f"IDs de cliente duplicados: {total_ids_duplicados}")

Valores ausentes: 0
Linhas duplicadas: 0
IDs de cliente duplicados: 0
